In [2]:
import os
import json
import pandas as pd

root_dir = "../result/maskable_ppo"          # 根目录
records = []

for dirpath, _, filenames in os.walk(root_dir):
    for fname in filenames:
        if not fname.endswith(".json"):
            continue
        fpath = os.path.join(dirpath, fname)

        # 1. 提取输入电路文件名
        # 文件名格式：Q=...-B=...-NS=...-E=0.json
        try:
            circuit_name = fname.split("Q=")[1].split("-")[0]
        except IndexError:
            # 如果文件名不符合预期，跳过或给出默认值
            circuit_name = None

        # 2. 读取 JSON 并提取指标
        try:
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)
            metrics = data.get("metrics", {})
            cx_ratio   = metrics.get("cx_ratio")
            depth_ratio = metrics.get("depth_ratio")
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
            continue

        records.append({
            "circuit": circuit_name,
            "cx_ratio": cx_ratio,
            "depth_ratio": depth_ratio
        })

# 3. 构建 DataFrame
df = pd.DataFrame(records)

# 4. 可选：保存到 CSV
# df.to_csv("summary.csv", index=False)

print(df.head())

                              circuit  cx_ratio  depth_ratio
0  20Q_gate_Tokyo_large_2_20_1.5_no.1  3.450382     3.255000
1   20Q_gate_Tokyo_large_2_3_1.5_no.7  2.794393     3.829787
2  20Q_gate_Tokyo_large_2_25_1.5_no.6  3.332481     2.807860
3   20Q_gate_Tokyo_large_1_3_1.5_no.6  2.892857     2.071429
4   20Q_gate_Tokyo_large_2_5_1.5_no.6  2.776923     2.657534


In [4]:
df.cx_ratio.mean(), df.depth_ratio.mean()  # 返回平均值

(np.float64(2.9325821534986773), np.float64(2.4709157002500683))

In [14]:
df_ha = pd.read_csv('../result/ha/ha_results.csv')
df_ha.head()

,hardware,depth_ratio,cx_ratio,init,circuit
0,sycamore,2.201835,8.189189,random,53Q_gate_Sycamore_small_2_3_1.5_no.7
1,rochester,3.206897,5.537815,sabre,53Q_gate_Rochester_small_1_3_1.5_no.7
2,tokyo,1.486486,1.400000,sa,20Q_gate_Tokyo_large_1_1_1.5_no.7
3,tokyo,1.203704,1.388889,sa,20Q_gate_Tokyo_large_2_1_1.5_no.4
4,rochester,4.142857,4.060914,sabre,53Q_gate_Rochester_large_2_5_1.5_no.1


In [16]:
df_ha = df_ha[(df_ha.hardware == 'tokyo') & (df_ha.init == 'sabre') & (df_ha.circuit.str.startswith('20Q_gate_Tokyo'))]
df_ha

,hardware,depth_ratio,cx_ratio,init,circuit
16,tokyo,1.874598,2.594320,sabre,20Q_gate_Tokyo_large_1_25_1.5_no.0
128,tokyo,2.108225,2.413876,sabre,20Q_gate_Tokyo_large_1_20_1.5_no.3
237,tokyo,1.780488,2.125000,sabre,20Q_gate_Tokyo_large_1_1_1.5_no.2
238,tokyo,2.052632,2.327434,sabre,20Q_gate_Tokyo_large_2_3_1.5_no.6
240,tokyo,1.458333,1.697183,sabre,20Q_gate_Tokyo_large_1_15_1.5_no.5
...,...,...,...,...,...
6240,tokyo,1.911111,2.153846,sabre,20Q_gate_Tokyo_large_2_2_1.5_no.9
6248,tokyo,2.243902,2.477612,sabre,20Q_gate_Tokyo_large_1_2_1.5_no.8
6269,tokyo,1.928571,2.178571,sabre,20Q_gate_Tokyo_large_1_1_1.5_no.9
6277,tokyo,2.102564,2.418182,sabre,20Q_gate_Tokyo_large_2_1_1.5_no.8


In [17]:
df_ha.cx_ratio.mean(), df_ha.depth_ratio.mean()  # 返回平均值

(np.float64(2.2655038717936264), np.float64(1.843160274399026))